# Generate tobacco and weed labels

Create unsplit pseudo-labels from a completed migration and an original or trained SAM 3 checkpoint. Tobacco comes from SAM 3. Weed comes from original three-class patch masks, or SAM 3 for binary full-image masks. These annotations are not independent ground truth and require visual review.


In [7]:
from pathlib import Path
import json
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'notebooks/reproduction.py').is_file())
sys.path.insert(0, str(ROOT / 'notebooks'))
from reproduction import new_run


In [8]:
RUN_LABELING = True
MIGRATION = ROOT / 'experiments/reproductions/tobacco-image-migration-04'
# CHECKPOINT = ROOT / 'experiments/reproductions/sam3-simple-training-67/best.pt'
CHECKPOINT = Path('C:/Users/natsusa_aki/academic/sam3/dam3/sam3.pt')
RUN_NAME = 'tobacco-pseudolabels-05'
CONFIDENCE = 0.6
PROMPT_MODE = 'reference_box'
REFERENCE_IMAGE = ROOT / 'data/sam3_reference/image_000001.png'
REFERENCE_BOX_XYWH = (300, 165, 175, 180)
TOBACCO_PROMPT = 'tobacco plant'


## Load the model

Weights are loaded locally with no download. Trained checkpoints reuse their saved resolution and epoch; original weights use resolution 1008 and have no training epoch. Inference requires CUDA and the SAM 3 environment used by the training notebook. Choose `reference_box` to use a positive tobacco box from the reference image, or `text` to use `TOBACCO_PROMPT`. The reference adapter uses experimental SAM 3 internal methods, as in notebook 06; inspect predictions before labeling. Weed inference still uses text.


In [9]:
def load_model(checkpoint):
    import torch
    from sam3.model_builder import build_sam3_image_model
    from collections import OrderedDict
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError('SAM 3 labeling requires a CUDA GPU with BF16 support.')
    payload = torch.load(checkpoint, map_location='cpu', weights_only=True)
    if not isinstance(payload, dict):
        raise ValueError('Select a SAM 3 checkpoint containing model weights.')
    resolution = payload['options']['resolution'] if 'options' in payload else 1008
    epoch = payload.get('epoch')
    state = payload.get('model', payload)
    if any(key.startswith('detector.') for key in state):
        state = {key.removeprefix('detector.'): value for key, value in state.items()
                 if key.startswith('detector.')}
    state = {key: value for key, value in state.items()
             if not key.startswith('backbone.vision_backbone.sam2_convs.')}
    model = build_sam3_image_model(device='cpu', eval_mode=True, load_from_HF=False,
                                  enable_segmentation=True, enable_inst_interactivity=False,
                                  compile=False)
    model.load_state_dict(state, strict=True)
    for block in model.backbone.vision_backbone.trunk.blocks:
        mlp = block.mlp
        block.mlp = torch.nn.Sequential(OrderedDict(
            (name, getattr(mlp, name)) for name in ('fc1', 'act', 'drop1', 'norm', 'fc2', 'drop2')))
    return model.cuda().eval(), resolution, epoch

def normalize_reference_box(box, image_size):
    import math
    if len(box) != 4:
        raise ValueError('REFERENCE_BOX_XYWH must contain x, y, width, and height.')
    x, y, width, height = box
    image_width, image_height = image_size
    if (not all(math.isfinite(value) for value in box) or
            image_width <= 0 or image_height <= 0 or x < 0 or y < 0 or
            width <= 0 or height <= 0 or
            x + width > image_width or y + height > image_height):
        raise ValueError('Reference box must be finite, positive, and inside the reference image.')
    return [(x + width / 2) / image_width, (y + height / 2) / image_height,
            width / image_width, height / image_height]

def encode_reference_prompt(processor, reference_image, box):
    normalized_box = normalize_reference_box(box, reference_image.size)
    state = processor.set_image(reference_image)
    state = processor.add_geometric_prompt(state=state, box=normalized_box, label=True)
    prompt, prompt_mask, _ = processor.model._encode_prompt(
        state['backbone_out'], processor.find_stage, state['geometric_prompt'])
    return prompt, prompt_mask

def predict_with_reference(processor, state, reference_prompt):
    from unittest.mock import patch
    state['geometric_prompt'] = processor.model._get_dummy_prompt()
    prompt, prompt_mask = reference_prompt

    def use_reference_prompt(backbone_out, find_input, geometric_prompt):
        return prompt, prompt_mask, backbone_out

    with patch.object(processor.model, '_encode_prompt', new=use_reference_prompt):
        return processor._forward_grounding(state)

def model_instances(output, shape, threshold):
    import numpy as np
    result = []
    for mask, score in zip(output['masks'], output['scores']):
        score = float(score)
        array = mask.detach().cpu().numpy()
        if array.size != shape[0] * shape[1]:
            raise ValueError(f'SAM mask dimensions do not match image: {array.shape}, {shape}')
        array = array.reshape(shape).astype(bool)
        if np.isfinite(score) and score >= threshold and array.any():
            result.append((score, array))
    return result


## Assemble labels

SAM masks are sorted by confidence within each class; tobacco wins class overlaps. For three-class source masks, each remaining 8-connected weed region is an instance. For binary sources, weed instances come from SAM. COCO RLE preserves holes and disconnected parts. YOLO boxes enclose final tobacco pixels.


In [10]:
def assemble_labels(shape, tobacco, weed, original_mask=None):
    import numpy as np
    from scipy.ndimage import label
    semantic = np.zeros(shape, dtype=np.uint8)
    instances = []
    for category, predictions in ((1, tobacco), (2, weed if original_mask is None else [])):
        for score, mask in sorted(predictions, key=lambda item: -item[0]):
            mask = np.asarray(mask, dtype=bool)
            if mask.shape != shape:
                raise ValueError('Prediction mask dimensions differ from image.')
            remaining = mask & (semantic == 0)
            if remaining.any():
                semantic[remaining] = category
                instances.append({'category_id': category, 'mask': remaining, 'score': float(score)})
    if original_mask is not None:
        original_mask = np.asarray(original_mask)
        if original_mask.shape != shape or not np.isin(original_mask, [0, 1, 2]).all():
            raise ValueError('Expected a same-size three-class source mask.')
        remaining = (original_mask == 2) & (semantic == 0)
        components, count = label(remaining, structure=np.ones((3, 3), dtype=np.uint8))
        for component in range(1, count + 1):
            mask = components == component
            semantic[mask] = 2
            instances.append({'category_id': 2, 'mask': mask, 'score': None})
    return semantic, instances

def encode_instances(image_id, instances, width, height, next_id):
    import numpy as np
    from pycocotools import mask as mask_api
    annotations, boxes = [], []
    for instance in instances:
        mask = np.asarray(instance['mask'], dtype=np.uint8)
        if mask.shape != (height, width) or not mask.any():
            raise ValueError('Instance mask must be nonempty and match the image.')
        rle = mask_api.encode(np.asfortranarray(mask))
        area = int(mask_api.area(rle))
        x, y, box_width, box_height = [float(value) for value in mask_api.toBbox(rle)]
        rle['counts'] = rle['counts'].decode('ascii')
        annotation = {'id': next_id + len(annotations), 'image_id': image_id,
                      'category_id': instance['category_id'], 'segmentation': rle,
                      'area': area, 'bbox': [x, y, box_width, box_height], 'iscrowd': 0}
        if instance['score'] is not None:
            annotation['score'] = instance['score']
        annotations.append(annotation)
        if instance['category_id'] == 1:
            boxes.append(f'0 {(x + box_width / 2) / width:.8f} {(y + box_height / 2) / height:.8f} '
                         f'{box_width / width:.8f} {box_height / height:.8f}')
    return annotations, boxes


## Generate the dataset

The output has `images/`, `semantic_masks/`, `labels/` (YOLO boxes), `annotations/instances.json` (COCO), and provenance. The progress bar counts completed images and estimates the remaining time. Images remain unsplit. Failed runs are incomplete: use a new run name to retry.


In [11]:
def generate_dataset(migration, run, processor, predict, config):
    import hashlib
    import shutil
    import numpy as np
    from PIL import Image
    from tqdm.auto import tqdm
    migration, run = Path(migration), Path(run)
    summary = json.loads((migration / 'summary.json').read_text(encoding='utf-8'))
    manifest = json.loads((migration / 'manifest.json').read_text(encoding='utf-8'))
    if summary.get('status') != 'complete' or summary.get('images') != len(manifest['images']):
        raise ValueError('Migration is incomplete or its manifest is inconsistent.')
    for folder in ('images', 'semantic_masks', 'labels', 'annotations'):
        (run / folder).mkdir()
    images, annotations = [], []
    for record in tqdm(manifest['images'], desc='Labeling images', unit='image'):
        name = record['file_name']
        if Path(name).name != name or not name.endswith('.png'):
            raise ValueError(f'Invalid migrated image name: {name}')
        image_path = migration / 'images' / name
        mask_path = migration / 'source_masks' / name
        if hashlib.sha256(mask_path.read_bytes()).hexdigest() != record['sources'][0]['mask_sha256']:
            raise ValueError(f'Migrated source mask changed: {mask_path}')
        with Image.open(image_path) as source:
            image = source.convert('RGB')
        rgb = np.asarray(image)
        digest = hashlib.sha256(str(rgb.shape).encode() + rgb.tobytes()).hexdigest()
        if digest != record['rgb_sha256'] or image.size != (record['width'], record['height']):
            raise ValueError(f'Migrated image changed: {image_path}')
        with Image.open(mask_path) as source:
            original_mask = np.asarray(source)
        if original_mask.shape != rgb.shape[:2]:
            raise ValueError(f'Migrated mask dimensions changed: {mask_path}')
        if record['mask_type'] == 'three_class':
            weed_hash = hashlib.sha256((original_mask == 2).tobytes()).hexdigest()
            if weed_hash != record['weed_sha256']:
                raise ValueError(f'Migrated weed mask changed: {mask_path}')
        elif record['mask_type'] != 'binary':
            raise ValueError(f'Unknown source mask type: {record["mask_type"]}')
        tobacco, weed = predict(processor, image, record['mask_type'], config['confidence'])
        semantic, instances = assemble_labels(rgb.shape[:2], tobacco, weed,
            original_mask if record['mask_type'] == 'three_class' else None)
        encoded, boxes = encode_instances(record['id'], instances, image.width, image.height,
                                           len(annotations) + 1)
        annotations.extend(encoded)
        images.append({'id': record['id'], 'file_name': f'images/{name}',
                       'width': image.width, 'height': image.height})
        shutil.copyfile(image_path, run / 'images' / name)
        Image.fromarray(semantic).save(run / 'semantic_masks' / name)
        (run / 'labels' / f'{Path(name).stem}.txt').write_text(
            '\n'.join(boxes) + ('\n' if boxes else ''), encoding='utf-8')
    coco = {'images': images, 'annotations': annotations,
            'categories': [{'id': 1, 'name': 'tobacco'}, {'id': 2, 'name': 'weed'}]}
    (run / 'annotations' / 'instances.json').write_text(json.dumps(coco), encoding='utf-8')
    (run / 'migration_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    (run / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    (run / 'summary.json').write_text(json.dumps({'status': 'complete', 'images': len(images),
        'annotations': len(annotations)}, indent=2), encoding='utf-8')
    return coco

def predict_image(processor, image, mask_type, threshold, tobacco_prompt='tobacco plant',
                  reference_prompt=None):
    shape = (image.height, image.width)
    state = processor.set_image(image)
    if reference_prompt is None:
        processor.reset_all_prompts(state)
        output = processor.set_text_prompt(prompt=tobacco_prompt, state=state)
    else:
        output = predict_with_reference(processor, state, reference_prompt)
    tobacco = model_instances(output, shape, threshold)
    weed = []
    if mask_type == 'binary':
        processor.reset_all_prompts(state)
        output = processor.set_text_prompt(prompt='weed plant', state=state)
        weed = model_instances(output, shape, threshold)
    return tobacco, weed


In [12]:
if RUN_LABELING:
    from functools import partial
    import hashlib
    import importlib.metadata
    import io
    import platform
    from PIL import Image
    import torch
    from sam3.model.sam3_image_processor import Sam3Processor
    if not 0 <= CONFIDENCE <= 1:
        raise ValueError('CONFIDENCE must be between 0 and 1.')
    if PROMPT_MODE not in ('reference_box', 'text'):
        raise ValueError("PROMPT_MODE must be 'reference_box' or 'text'.")
    if not CHECKPOINT.is_file():
        raise FileNotFoundError(f'Trained SAM 3 checkpoint not found: {CHECKPOINT}')
    reference_image = None
    reference_config = None
    if PROMPT_MODE == 'reference_box':
        if not REFERENCE_IMAGE.is_file():
            raise FileNotFoundError(f'Reference image not found: {REFERENCE_IMAGE}')
        reference_bytes = REFERENCE_IMAGE.read_bytes()
        with Image.open(io.BytesIO(reference_bytes)) as source:
            reference_image = source.convert('RGB')
        normalize_reference_box(REFERENCE_BOX_XYWH, reference_image.size)
        reference_config = {'image': str(REFERENCE_IMAGE.resolve()),
                            'image_sha256': hashlib.sha256(reference_bytes).hexdigest(),
                            'image_size': list(reference_image.size),
                            'box_xywh': list(REFERENCE_BOX_XYWH)}
    migration_summary = json.loads((MIGRATION / 'summary.json').read_text(encoding='utf-8'))
    if migration_summary.get('status') != 'complete':
        raise ValueError('Migration is not complete.')
    model, resolution, epoch = load_model(CHECKPOINT)
    processor = Sam3Processor(model, resolution=resolution, confidence_threshold=CONFIDENCE)
    with CHECKPOINT.open('rb') as handle:
        checkpoint_hash = hashlib.file_digest(handle, 'sha256').hexdigest()
    config = {'migration': str(MIGRATION.resolve()), 'checkpoint': str(CHECKPOINT.resolve()),
              'checkpoint_sha256': checkpoint_hash, 'checkpoint_epoch': epoch,
              'resolution': resolution, 'confidence': CONFIDENCE,
              'tobacco_prompt_mode': PROMPT_MODE,
              'tobacco_prompt': TOBACCO_PROMPT if PROMPT_MODE == 'text' else None,
              'tobacco_reference': reference_config, 'weed_prompt': 'weed plant',
              'weed_source': 'original three-class masks; SAM 3 for binary masks',
              'python': sys.version, 'platform': platform.platform(),
              'torch': str(torch.__version__), 'cuda': torch.version.cuda,
              'gpu': torch.cuda.get_device_name(),
              'packages': {item.metadata['Name']: item.version for item in importlib.metadata.distributions()}}
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
        reference_prompt = (encode_reference_prompt(processor, reference_image, REFERENCE_BOX_XYWH)
                            if reference_image is not None else None)
        predict = partial(predict_image, tobacco_prompt=TOBACCO_PROMPT,
                          reference_prompt=reference_prompt)
        label_run = new_run(ROOT, RUN_NAME)
        coco = generate_dataset(MIGRATION, label_run, processor, predict, config)
    print(f"Labeled {len(coco['images'])} images in {label_run}")
else:
    print('Labeling disabled. Set MIGRATION, CHECKPOINT, RUN_NAME, and RUN_LABELING to enable it.')


Labeling images:   0%|          | 0/2508 [00:00<?, ?image/s]

Labeled 2508 images in c:\Users\natsusa_aki\academic\tobacco-article\experiments\reproductions\tobacco-pseudolabels-05
